# 01 — Dataset preparation and composition descriptors
Prepare CIF metadata, centrosymmetry labels, chemical domains, composition groups and 132 Magpie descriptors. Configure `private_paths.py` before running this notebook.

**Inputs:** source CIF files, or the three intermediate tables listed below. **Outputs:** the selected project's `datasets/` tables and `datasets/reports/` audit records. Notebook 02 subsequently generates group-aware splits.

For numerical reproduction of PAPER_V2, retain the archived feature and split files and start at notebook 03. COD_CANON_V2 reused historical metadata and symmetry labels; MP_ALL retained its historical datasets and splits. Running the current preparation pipeline on a new CIF snapshot, especially MP, is a new dataset build and does not establish equivalence to those archived results. See [reproduction inputs](../docs/REPRODUCIBILITY.md).


In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipelines.data_prep_pipeline import DataPrepPipeline
from pipelines.features_pipeline import FeaturesPipeline
from targets.centrosymmetry import CentrosymmetryLabeler
from project_config import make_cfg
from private_paths import COD_DIR, MP_DIR

PROJECT_NAME = "COD_CANON_V2"  # Choose MP_ALL to prepare MP with the current pipeline.
ROOT_CIF = {"COD_CANON_V2": COD_DIR, "MP_ALL": MP_DIR}[PROJECT_NAME]
cfg = make_cfg(project_name=PROJECT_NAME, root_cif=ROOT_CIF,
               raw_sample_n=None, raw_sample_seed=None)
PREPARATION_MODE = "from_cif"  # "from_cif" or "resume"
SOURCE_DATASETS = None  # Required only for resume: directory of saved intermediate tables.
FEATURE_STEPS = ("magpie",)
FORCE_FEATURES = False
REBUILD_GROUPS = False
if PREPARATION_MODE not in {"from_cif", "resume"}:
    raise ValueError("Choose from_cif or resume.")


## Full preparation from CIF files
With `PREPARATION_MODE="from_cif"`, steps 1–8 list and optionally sample CIF files, extract metadata, assign symmetry labels and chemical classes, build composition groups, assemble the master and filter conflicts. The default retains the complete input listing. Use a separate project directory when preparing a new snapshot to preserve archived inputs.


In [ ]:
if PREPARATION_MODE == "from_cif":
    prep = DataPrepPipeline(cfg, labeler=CentrosymmetryLabeler(cfg))
    prep.run(from_step=1, to_step=8)


## Resume from intermediate tables
With `PREPARATION_MODE="resume"`, set `SOURCE_DATASETS` to a directory containing `dataset_raw_cif_listing.csv`, `dataset_metadata_cif.csv` and `dataset_symmetry_labels.csv`. Existing destination copies must be identical. Available feature caches are copied only when absent.

Chemical classes are recomputed. Composition groups are reused only when present and `REBUILD_GROUPS=False`; set it to `True` if metadata, labels or grouping rules have changed. Masters and filtered datasets are rebuilt. Historical masters, groups and folds are not copied from the source directory.


In [ ]:
if PREPARATION_MODE == "resume":
    if SOURCE_DATASETS is None:
        raise ValueError("Set SOURCE_DATASETS to the existing intermediate-table directory.")
    SOURCE_DATASETS = Path(SOURCE_DATASETS)
    from shutil import copy2
    from filecmp import cmp

    required_inputs = {
        "dataset_raw_cif_listing.csv": cfg.DATASET_RAW_CIF_LISTING,
        "dataset_metadata_cif.csv": cfg.DATASET_METADATA_CIF,
        "dataset_symmetry_labels.csv": cfg.DATASET_SYMMETRY_LABELS,
    }
    missing = [str(SOURCE_DATASETS / name) for name in required_inputs
               if not (SOURCE_DATASETS / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing saved intermediate inputs: {missing}")
    for name, destination in required_inputs.items():
        source, target = SOURCE_DATASETS / name, Path(destination)
        if target.exists() and not cmp(source, target, shallow=False):
            raise ValueError(f"Destination input differs from the selected source: {target}")
    for name, destination in required_inputs.items():
        target = Path(destination)
        if not target.exists():
            copy2(SOURCE_DATASETS / name, target)

    cache_paths = {"magpie": cfg.DATASET_FEATURES, "simple": cfg.DATASET_FEATURES_SIMPLE,
                   "frac": cfg.DATASET_FEATURES_FRAC}
    for feature_step in FEATURE_STEPS:
        target = Path(cache_paths[feature_step])
        source = SOURCE_DATASETS / target.name
        if source.is_file() and not target.exists():
            copy2(source, target)

    prep = DataPrepPipeline(cfg, labeler=CentrosymmetryLabeler(cfg))
    prep_steps = [5]
    if REBUILD_GROUPS or not Path(cfg.DATASET_GROUPS).is_file():
        prep_steps.append(6)
    prep_steps.extend([7, 8])
    prep.run(steps=prep_steps)


## Group validation

Step 6 groups exact proportional compositions first, then applies an absolute tolerance
of `1e-5` to atomic fractions with identical element support. Step 7 propagates the canonical
group columns and original labels to the master. Step 8 removes entire C/N conflict groups,
symmetry conflicts, missing groups and invalid labels.

Step 8 checks IDs, group aliases, exact-key assignment, match types, tolerance consistency,
row balance and complete removal of C/N conflicts before writing the cleaned master.


## Feature generation
Both preparation paths feed the same feature stage. A missing cache generates features for all records; a valid cache reuses matching IDs and generates missing records. Set `FORCE_FEATURES=True` to regenerate the full feature table. Schema, formula identity and finite feature values are checked. Records without complete finite descriptors are excluded with an explicit report.

The logger records parameters, progress, exceptions and elapsed time under the project's `logs/` directory. Generated descriptors and refreshed group metadata are saved together.


In [ ]:
from core.notebook_logger import NotebookLogger
import time
import traceback

feat = FeaturesPipeline(cfg)
feature_logger = NotebookLogger(
    project_dir=cfg.PROJECT_DIR,
    prefix=f"{cfg.PROJECT_NAME}_features",
    timestamp_fmt="%Y%m%d_%H%M%S_%f",
)
feature_logger.start(header_lines=[
    f"Project: {cfg.PROJECT_NAME}",
    "Pipeline: FeaturesPipeline",
    f"Input cleaned master: {cfg.DATASET_CLEANED_MASTER}",
    f"Output datasets: {cfg.DATASETS_DIR}",
    f"Steps: {FEATURE_STEPS}; aggregate=True; force={FORCE_FEATURES}",
])
feature_started = time.perf_counter()
try:
    feat.run(steps=FEATURE_STEPS, aggregate=True, force=FORCE_FEATURES)
except BaseException:
    traceback.print_exc()
    raise
finally:
    print(f"Feature stage elapsed_seconds: {time.perf_counter() - feature_started:.3f}")
    feature_logger.stop()


## Reports and Methods

Step 8 writes reports under `Path(cfg.DATASETS_DIR) / "reports"`
(for the current project: `COD_CANON_V2/datasets/reports/`):

- `canonicalization_report.csv`: one row per group, exact merges, tolerance components, examples and before/after counts.
- `conflicting_groups.csv`: all records from groups containing both C and N before filtering.
- `tolerance_matches.csv`: all members of tolerance components (not all directly matching pairs).
- `excluded_records.csv`: rejected IDs, one primary reason and all applicable reason flags.
- `dataset_summary.csv`: before/after counts, C/N counts and primary rejection reasons, overall and per chemical domain.
- `group_validation.csv`: explicit pass/fail checks.
- `parameters.txt`: tolerance, grouping/filter rules, master path and SHA-256.

The feature stage additionally writes `feature_exclusions_<step>.csv` and
`feature_summary_<step>.csv` in the same directory, accounting for subsequent feature exclusions.

Primary rejection reasons are exclusive and sum to the number removed. Additional flags may overlap.
A group can span domains; domain group counts must not be added to obtain the global total.
These reports cover master -> cleaned master. They do not validate train/test/fold separation.
In notebook 02, select the same project and the normal feature dataset (`dataset_features.csv` for Magpie).

### Methods text: composition grouping and filtering

Chemical classes are recomputed in step 5 using the current classifier, whose metal set
includes lanthanides and actinides. These class labels are propagated through the master
to the feature table; historical class labels stored in a feature cache are not reused.

Element coefficients are parsed from their decimal representations as exact rational numbers.
Exactly proportional compositions receive the same reduced integer key. Distinct exact keys with
identical element support are joined when the maximum absolute difference between their atomic
fractions is at most 10^-5, with zero relative tolerance. Connected components define the final
groups; endpoint compositions connected through intermediate matches may differ by more than
10^-5. Canonical group identifiers are propagated to the cleaned dataset and feature table.
Groups containing both centrosymmetric (C) and non-centrosymmetric (N) records are identified
on the complete master before record filtering, and all their members are excluded across
chemical domains. Records with inconsistent CIF/spglib symmetry, missing canonical groups or
invalid C/N labels are also excluded. Same-label records are retained. Existing feature values
may be reused after ID, formula, schema and finiteness checks, with group metadata
refreshed from the cleaned master. Missing cached records are processed by the existing
feature generator. Records for which valid finite features are not produced are excluded
from the corresponding feature dataset and recorded in a separate exclusion report.

Split/fold validation and the final record counts must be reported after notebook 02 has run.
